# 10年定着予測 - 却下済み特徴量ブロックの再検証（39_、低ノイズプロトコル）

**背景**: `37_` で判明した通り、この特徴量セットでの**シード分散は sd 0.0047〜0.0078**あり、
`20_`〜`35_` でブロックの採否を判定してきた効果量（L_v1→L_v2 の Public差 0.0002、N・O の ±0.001 など）
より大きい。つまり過去のアブレーションは**測定器の分解能を超えた判定**をしていた可能性がある。

さらに `37_` では、検証セットを18名（3.3%）変えただけで Optuna が全く別の領域に着地した
（depth 4→8、L2正則化が1/26）。`16_`〜`35_` は**構成ごとに Optuna を回し直していた**ため、
「ブロックの効果」と「Optunaの着地点のブレ」が交絡していた。

本ノートブックは、却下済みブロックを**ノイズを抑えたプロトコル**で測り直す。

## 旧プロトコルとの違い

| 項目 | `16_`〜`35_`（旧） | `39_`（新） |
|---|---|---|
| ハイパーパラメータ | **構成ごとにOptuna探索** | **`A_PARAMS`に固定** ← 交絡を除去 |
| シード | 1 | **5**（平均と分散の両方を報告） |
| 検証セット | 全体（早期退職者を含む） | **生存者のみ**（EDA v6第1〜2節） |
| split | 80/20・75/25 | 同左（据え置き） |

ハイパーパラメータを固定することで Optuna が不要になり、**旧プロトコルより安く**なる
（12構成 × 2 split × 5シード = 120回の学習、Optunaゼロ）。

## 検証する構成

現在の最良特徴量セット（`28_`/`37_` = ベースライン + ブロックL_v2）を基準に、単体で追加する。

**A群: 却下済みブロック**（旧プロトコルで不採用と判定されたもの）

| ブロック | 内容 | 旧判定 |
|---|---|---|
| F | 経験等級整合性（中途の前職経験月数からの残差） | ノイズ幅程度〜微悪化 |
| G | 自己学習（実施月数・時間・テーマ数） | 単体 Public 0.549606（ベースライン相当） |
| H | エンゲージメント深掘り（情報共有ゼロ月数・信頼度の本人内偏差） | Public 悪化 |
| I | 人物所見キーワード（柔軟性・主体性・計画性） | Public 悪化（ギャップ0.041） |
| K | 早期昇給タイミング | Public 悪化（0.569681） |
| M | 専攻×職種の分析的適合ミスマッチ | 検証で悪化、未提出 |
| N | 短期モメンタム比率 | 検証で悪化、未提出 |
| O | 活動密度比率 | 検証で悪化、未提出 |

**B群: 確認済みブロックの組み合わせ**（個別にはPublicで改善が確認されているのに、Lとの併用が未検証）

| 構成 | 根拠 |
|---|---|
| E + L | E単体 Public 0.534829、L単体 0.529454。両方とも18_ベースライン(0.550352)を明確に改善 |
| J + L | J単体 Public 0.540648。`27_`で combo_JL を検証したが「Lとほぼ同水準（差0.0003）」で提出見送り |
| E + J + L | 上記2つが効くなら |

> ⚠️ **B群には既知のリスクがある**: `combo_EFG`・`combo_EG`・`combo_EI` はいずれも検証で改善して見えて
> Public で悪化した。ただしそれらは F・G・I という**単体で弱い/有害なブロック**との組み合わせだった。
> E・J・L は**3つとも単体でPublicで確認済み**であり、状況が異なる。とはいえ検証だけで採用はしない。

## 期待される結末

EDA v6 第4節（効果量のホライズン依存性）から、**A群の行動系（G・H・K・N・O）は理論的に不利**である
——行動系が予測しているのは短期離職で、Testには短期離職者が1人もいない。したがって
「旧プロトコルがノイズで誤判定していた」のではなく「本当に効かない」可能性が高い。

本ノートブックの価値は、**A群が本当にダメだと低ノイズで確認して探索を打ち切ること**と、
**B群（未検証の有望な組み合わせ）を初めてまともな分解能で測ること**の2点にある。

## 判定手続き（`16_`〜`35_`からの最大の変更点）

このプロジェクトが繰り返し失敗してきたのは「検証で改善→Publicで悪化」であり、その原因は
**測定の分解能不足**だった。そこで計算資源を精度そのものに投じる。

| 項目 | `16_`〜`35_`（旧） | `39_`（新） |
|---|---|---|
| ハイパーパラメータ | 構成ごとにOptuna探索 | **`A_PARAMS`に固定**（交絡を除去） |
| split | 2（80/20, 75/25） | **4**（85/15, 80/20, 75/25, 70/30） |
| シード | 1 | **8** |
| 反復数 | early stopping（構成ごとにバラつく） | **固定**（`38_`の感度曲線の最適点を件数比でスケール） |
| 検証セット | 全体 | **生存者のみ** |
| 採否の判定 | 「両split一貫して改善」 | **ペアードブートストラップの95%信頼区間** |

**ペアードブートストラップ**が新しい要素。各構成とベースラインは*同一の検証社員*で評価されるので、
検証社員をリサンプリングして「構成 − ベースライン」のlogloss差の分布を直接求められる。
これにより「改善幅がシード分散と検証標本のゆらぎを超えているか」を、閾値の当てずっぽうではなく
**信頼区間で**判定できる。

`26_`のブロックKは「両split一貫して改善」を満たしながらPublicで3.5%悪化した。
一貫性だけでは足りないというのが、この変更の直接の動機である。

**反復数の固定**（`38_`の結果を反映）も重要な変更。`38_`の感度曲線で、early stoppingの
`best_iteration`（448.6）は固定反復数の真の最適点（560）を系統的に下回ることが分かった。
アブレーションでearly stoppingを使うと構成ごとに停止位置がバラつき、
「ブロックの効果」と「反復数の選ばれ方」が交絡する。固定すれば、構成間で変わるのは
**特徴量ブロックだけ**になる。曲線は350〜900で平坦（振れ幅0.0035 < シードsd 0.0044）なので、
学習件数に比例させたスケーリングで安全。

## 実行環境

Google Colab Pro の **CPUハイメモリ**。12構成 × 4 split × 8シード = **384回の学習**（Optunaはゼロ）。
`37_`の実測（1回あたり約5秒）から **40〜60分** を想定。
ローカルMacで先行実行しないこと（チェックポイントのDrive同期事故を避けるため）。

In [1]:
!pip install -q catboost optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 28.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 20.4 MB/s eta 0:00:00


In [2]:
import multiprocessing
print(f"CPUコア数: {multiprocessing.cpu_count()}（今回はCPUで学習するため、GPUランタイムは不要）")

CPUコア数: 8（今回はCPUで学習するため、GPUランタイムは不要）


In [3]:
import sys
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path("/content/drive/MyDrive/jaggle_2026")
sys.path.append(str(PROJECT_ROOT))

Mounted at /content/drive


In [4]:
import datetime
import json
import re
import warnings

import numpy as np
import pandas as pd
import catboost as cb
import optuna
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import log_loss
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

from common.utils.logger import get_logger
from common.utils.metrics import calculate_logloss
from common.utils.seed import seed_everything

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
seed_everything(seed=SEED)

TARGET_COL = "10年定着ラベル"
ID_COL = "社員ID"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [5]:
SCRIPT_NAME = "39_rejected_blocks_low_noise_recheck"
TODAY = datetime.datetime.now().strftime("%Y%m%d")

LOG_DIR = PROJECT_ROOT / "logs"
logger = get_logger(SCRIPT_NAME, log_dir=str(LOG_DIR))
logger.info(f"=== [{SCRIPT_NAME}] 実験開始 ===")

OUTPUT_DIR = PROJECT_ROOT / "data" / "output" / TODAY
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAVED_MODELS_DIR = PROJECT_ROOT / "saved_models" / TODAY / SCRIPT_NAME
SAVED_MODELS_DIR.mkdir(parents=True, exist_ok=True)

# チェックポイント（日付非依存の固定パス。セッションをまたいで再開できるようにする）
CHECKPOINT_DIR = PROJECT_ROOT / "data" / "output" / "_checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = CHECKPOINT_DIR / f"{SCRIPT_NAME}_checkpoint.csv"

RESET_CHECKPOINT = True  # Trueにすると既存チェックポイントを削除して最初から再計算する
if RESET_CHECKPOINT and CHECKPOINT_PATH.exists():
    CHECKPOINT_PATH.unlink()
    print("チェックポイントを削除しました（全構成を再計算します）")

logger.info(f"Output Directory: {OUTPUT_DIR}")
logger.info(f"Checkpoint Path: {CHECKPOINT_PATH}")
if CHECKPOINT_PATH.exists():
    logger.info(f"既存のチェックポイントを発見: {len(pd.read_csv(CHECKPOINT_PATH))}件の結果が記録済み")
else:
    logger.info("チェックポイントは未作成（新規実行）")

[2026-08-11 15:35:04] [INFO] === [39_rejected_blocks_low_noise_recheck] 実験開始 ===


INFO:39_rejected_blocks_low_noise_recheck:=== [39_rejected_blocks_low_noise_recheck] 実験開始 ===


[2026-08-11 15:35:05] [INFO] Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260811


INFO:39_rejected_blocks_low_noise_recheck:Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260811


[2026-08-11 15:35:05] [INFO] Checkpoint Path: /content/drive/MyDrive/jaggle_2026/data/output/_checkpoints/39_rejected_blocks_low_noise_recheck_checkpoint.csv


INFO:39_rejected_blocks_low_noise_recheck:Checkpoint Path: /content/drive/MyDrive/jaggle_2026/data/output/_checkpoints/39_rejected_blocks_low_noise_recheck_checkpoint.csv


[2026-08-11 15:35:05] [INFO] チェックポイントは未作成（新規実行）


INFO:39_rejected_blocks_low_noise_recheck:チェックポイントは未作成（新規実行）


In [6]:
INPUT_DIR = PROJECT_ROOT / "data" / "input"

train_persona = pd.read_csv(INPUT_DIR / "employee_persona_train.csv")
test_persona = pd.read_csv(INPUT_DIR / "employee_persona_test.csv")
train_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_train.csv")
test_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_test.csv")

logger.info(f"Train Persona Shape: {train_persona.shape}, Test Persona Shape: {test_persona.shape}")
logger.info(f"Train Monthly Shape: {train_monthly.shape}, Test Monthly Shape: {test_monthly.shape}")

y_train = train_persona[TARGET_COL]
train_ids = train_persona[ID_COL].values
test_ids = test_persona[ID_COL].values

logger.info(f"定着率: {y_train.mean():.4f}")
logger.info(f"Train IDs: {len(train_ids)}, Test IDs: {len(test_ids)}")

[2026-08-11 15:35:10] [INFO] Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


INFO:39_rejected_blocks_low_noise_recheck:Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


[2026-08-11 15:35:10] [INFO] Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


INFO:39_rejected_blocks_low_noise_recheck:Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


[2026-08-11 15:35:10] [INFO] 定着率: 0.5647


INFO:39_rejected_blocks_low_noise_recheck:定着率: 0.5647


[2026-08-11 15:35:10] [INFO] Train IDs: 2761, Test IDs: 2502


INFO:39_rejected_blocks_low_noise_recheck:Train IDs: 2761, Test IDs: 2502


## 0. 早期退職者の特定（改善3の前提）

`月末在籍状態 == "退職"` の行を持つ社員が、0-23ヶ月の観測期間中に退職した社員。
Train に129名（全員ラベル0）、**Test には0名**。

In [7]:
EARLY_LEAVER_IDS = set(train_monthly.loc[train_monthly["月末在籍状態"] == "退職", ID_COL].unique())
_test_early = set(test_monthly.loc[test_monthly["月末在籍状態"] == "退職", ID_COL].unique())

logger.info(f"Train 早期退職者: {len(EARLY_LEAVER_IDS)}名 / {len(train_ids)}名 ({len(EARLY_LEAVER_IDS)/len(train_ids):.1%})")
logger.info(f"Test  早期退職者: {len(_test_early)}名 / {len(test_ids)}名")
_y_idx = train_persona.set_index(ID_COL)[TARGET_COL]
logger.info(f"早期退職者のラベル平均: {_y_idx.loc[list(EARLY_LEAVER_IDS)].mean():.4f}（0.0のはず）")
logger.info(f"定着率: 全体 {y_train.mean():.4f} / 早期退職者を除く {_y_idx[~_y_idx.index.isin(EARLY_LEAVER_IDS)].mean():.4f}")
assert len(_test_early) == 0, "Testに早期退職者が存在する。EDA v6の前提が崩れているので調査すること"

[2026-08-11 15:35:10] [INFO] Train 早期退職者: 129名 / 2761名 (4.7%)


INFO:39_rejected_blocks_low_noise_recheck:Train 早期退職者: 129名 / 2761名 (4.7%)


[2026-08-11 15:35:10] [INFO] Test  早期退職者: 0名 / 2502名


INFO:39_rejected_blocks_low_noise_recheck:Test  早期退職者: 0名 / 2502名


[2026-08-11 15:35:10] [INFO] 早期退職者のラベル平均: 0.0000（0.0のはず）


INFO:39_rejected_blocks_low_noise_recheck:早期退職者のラベル平均: 0.0000（0.0のはず）


[2026-08-11 15:35:10] [INFO] 定着率: 全体 0.5647 / 早期退職者を除く 0.5923


INFO:39_rejected_blocks_low_noise_recheck:定着率: 全体 0.5647 / 早期退職者を除く 0.5923


## 1. 基本特徴量関数の定義（split非依存、`18_`〜`26_`と同一ロジック）

In [8]:
def create_monthly_aggregation_features(monthly_df, employee_ids):
    """月次データから集約特徴量を生成（12_〜18_と同一ロジック）"""
    numeric_cols = [
        "残業時間", "有給取得日数", "欠勤日数", "研修時間",
        "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
        "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
        "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
        "顧客満足度評価", "担当プロジェクト数", "月例給与_円"
    ]

    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].copy()
        emp_data = emp_data.sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        for col in numeric_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            valid_values = values[~pd.isna(values)]

            features[f"{col}_mean"] = np.mean(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_std"] = np.std(valid_values) if len(valid_values) > 1 else np.nan
            features[f"{col}_min"] = np.min(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_max"] = np.max(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_median"] = np.median(valid_values) if len(valid_values) > 0 else np.nan
            mean_val = features[f"{col}_mean"]
            std_val = features[f"{col}_std"]
            features[f"{col}_cv"] = std_val / mean_val if (mean_val and mean_val != 0) else np.nan

            early = emp_data[emp_data["経過月数"].between(0, 2)][col]
            mid = emp_data[emp_data["経過月数"].between(3, 11)][col]
            late = emp_data[emp_data["経過月数"].between(12, 23)][col]
            features[f"{col}_early_mean"] = early.mean()
            features[f"{col}_mid_mean"] = mid.mean()
            features[f"{col}_late_mean"] = late.mean()
            features[f"{col}_late_minus_early"] = late.mean() - early.mean()
            features[f"{col}_late_early_ratio"] = (
                late.mean() / early.mean() if early.mean() and early.mean() != 0 else np.nan
            )

            if len(valid_values) >= 2:
                valid_indices = np.where(~pd.isna(values))[0]
                if len(valid_indices) >= 2:
                    slope, _, _, _, _ = stats.linregress(valid_indices, valid_values)
                    features[f"{col}_slope"] = slope
                else:
                    features[f"{col}_slope"] = np.nan
                first_val, last_val = valid_values[0], valid_values[-1]
                features[f"{col}_diff"] = last_val - first_val
                features[f"{col}_ratio"] = last_val / first_val if first_val != 0 else np.nan
            else:
                features[f"{col}_slope"] = features[f"{col}_diff"] = features[f"{col}_ratio"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_monthly_categorical_change_features(monthly_df, employee_ids):
    categorical_cols = ["部署ID", "職種", "役割", "等級", "勤務地", "上司ID"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in categorical_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            changes = sum(1 for i in range(1, len(values)) if pd.notna(values[i]) and pd.notna(values[i-1]) and values[i] != values[i-1])
            features[f"{col}_changes"] = changes
            features[f"{col}_unique_count"] = len(pd.Series(values).dropna().unique())
        if "月末在籍状態" in emp_data.columns:
            status_values = emp_data["月末在籍状態"].values
            features["leave_of_absence_flag"] = int("休職" in status_values)
            features["leave_of_absence_months"] = np.sum(status_values == "休職")
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_missing_value_features(monthly_df, employee_ids):
    missing_target_cols = ["360度評価_親和度", "360度評価_信頼度", "顧客満足度評価", "担当プロジェクト数"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in missing_target_cols:
            if col in emp_data.columns:
                values = emp_data[col].values
                total_months = len(values)
                features[f"{col}_missing_rate"] = pd.isna(values).sum() / total_months if total_months > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_domain_knowledge_features(monthly_df, employee_ids):
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
        eval_mean_list = [emp_data[col].mean() for col in eval_cols if col in emp_data.columns]
        features["engagement_score"] = np.nanmean(eval_mean_list) if len(eval_mean_list) > 0 else np.nan
        if "残業時間" in emp_data.columns:
            features["overtime_stability"] = emp_data["残業時間"].std()
        if "研修時間" in emp_data.columns and "残業時間" in emp_data.columns:
            training_mean = emp_data["研修時間"].mean()
            overtime_mean = emp_data["残業時間"].mean()
            features["training_overtime_ratio"] = training_mean / overtime_mean if overtime_mean > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_advanced_statistical_features(monthly_df, employee_ids):
    """統計的特徴量：歪度、尖度、パーセンタイル"""
    numeric_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in numeric_cols:
            if col in emp_data.columns:
                values = emp_data[col].dropna().values
                if len(values) >= 3:
                    features[f"{col}_skew"] = stats.skew(values)
                    features[f"{col}_kurtosis"] = stats.kurtosis(values)
                    features[f"{col}_q25"] = np.percentile(values, 25)
                    features[f"{col}_q75"] = np.percentile(values, 75)
                    features[f"{col}_iqr"] = features[f"{col}_q75"] - features[f"{col}_q25"]
                else:
                    features[f"{col}_skew"] = features[f"{col}_kurtosis"] = np.nan
                    features[f"{col}_q25"] = features[f"{col}_q75"] = features[f"{col}_iqr"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_cluster_features(monthly_df, employee_ids, n_clusters=5, seed=42):
    """クラスター特徴量：月次データの平均をKMeansクラスタリング"""
    key_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    agg_data = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id]
        row = {"社員ID": employee_id}
        for col in key_cols:
            if col in emp_data.columns:
                row[col] = emp_data[col].mean()
        agg_data.append(row)
    agg_df = pd.DataFrame(agg_data)
    feature_cols = [c for c in key_cols if c in agg_df.columns]
    X = agg_df[feature_cols].fillna(-999)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    kmeans = KMeans(n_clusters=n_clusters, random_state=seed, n_init=10)
    agg_df["cluster"] = kmeans.fit_predict(X_scaled)
    return agg_df[["社員ID", "cluster"]]


def create_eda_driven_features(monthly_df, employee_ids):
    """欠勤日数パターン・360度評価タイミング・月次ボラティリティ・比率特徴量（12_〜18_と同一ロジック）"""
    eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        absence_vals = emp_data["欠勤日数"].values
        nonzero = absence_vals > 0
        features["欠勤発生月数"] = int(nonzero.sum())
        max_run = cur_run = 0
        for v in nonzero:
            cur_run = cur_run + 1 if v else 0
            max_run = max(max_run, cur_run)
        features["欠勤_最長連続月数"] = max_run
        features["欠勤_連続フラグ"] = int(max_run >= 2)

        flagged = emp_data[emp_data["360度評価更新フラグ"] == 1]
        first_month = flagged["経過月数"].min() if len(flagged) > 0 else np.nan
        features["初回評価月"] = first_month
        features["is_早期評価"] = int(first_month <= 4) if pd.notna(first_month) else 0
        features["is_遅延評価"] = int(first_month >= 7) if pd.notna(first_month) else 0
        features["評価遅延度"] = abs(first_month - 5) if pd.notna(first_month) else np.nan

        for col in ["残業時間", "月例給与_円"]:
            vals = emp_data[col].dropna().values
            features[f"{col}_volatility"] = np.mean(np.abs(np.diff(vals))) if len(vals) >= 2 else np.nan

        n_months = len(emp_data)
        features["有給取得率"] = emp_data["有給取得日数"].sum() / n_months if n_months > 0 else np.nan
        features["評価項目間ばらつき"] = emp_data[eval_cols].std(axis=1).mean()

        features_list.append(features)
    return pd.DataFrame(features_list)


def create_manager_team_size_features(monthly_df, employee_ids):
    """初期（経過月数=0）時点で同じ上司IDを持つ社員数（12_〜18_と同一ロジック）"""
    month0 = monthly_df[monthly_df["経過月数"] == 0].copy()
    month0["初期上司_部下数"] = month0.groupby("上司ID")["社員ID"].transform("count")
    out = month0[["社員ID", "初期上司_部下数"]]
    return out[out["社員ID"].isin(employee_ids)].reset_index(drop=True)

print("✅ split非依存の基本特徴量関数定義完了")

✅ split非依存の基本特徴量関数定義完了


In [9]:
logger.info("-" * 60)
logger.info("split非依存の基本特徴量を生成中...")
logger.info("-" * 60)

train_monthly_agg = create_monthly_aggregation_features(train_monthly, train_ids)
test_monthly_agg = create_monthly_aggregation_features(test_monthly, test_ids)

train_cat_change = create_monthly_categorical_change_features(train_monthly, train_ids)
test_cat_change = create_monthly_categorical_change_features(test_monthly, test_ids)

train_missing = create_missing_value_features(train_monthly, train_ids)
test_missing = create_missing_value_features(test_monthly, test_ids)

train_domain = create_domain_knowledge_features(train_monthly, train_ids)
test_domain = create_domain_knowledge_features(test_monthly, test_ids)

train_advanced_stats = create_advanced_statistical_features(train_monthly, train_ids)
test_advanced_stats = create_advanced_statistical_features(test_monthly, test_ids)

train_cluster = create_cluster_features(train_monthly, train_ids, n_clusters=5, seed=SEED)
test_cluster = create_cluster_features(test_monthly, test_ids, n_clusters=5, seed=SEED)

train_eda_feats = create_eda_driven_features(train_monthly, train_ids)
test_eda_feats = create_eda_driven_features(test_monthly, test_ids)

train_mgr = create_manager_team_size_features(train_monthly, train_ids)
test_mgr = create_manager_team_size_features(test_monthly, test_ids)

logger.info("split非依存の基本特徴量生成完了")

[2026-08-11 15:35:10] [INFO] ------------------------------------------------------------


INFO:39_rejected_blocks_low_noise_recheck:------------------------------------------------------------


[2026-08-11 15:35:10] [INFO] split非依存の基本特徴量を生成中...


INFO:39_rejected_blocks_low_noise_recheck:split非依存の基本特徴量を生成中...


[2026-08-11 15:35:10] [INFO] ------------------------------------------------------------


INFO:39_rejected_blocks_low_noise_recheck:------------------------------------------------------------


[2026-08-11 15:40:40] [INFO] split非依存の基本特徴量生成完了


INFO:39_rejected_blocks_low_noise_recheck:split非依存の基本特徴量生成完了


## 2. テキストTF-IDF（A_v1、`18_`と同一・継続採用）

In [10]:
def create_tfidf_svd_features(train_persona, test_persona, col, max_features=300, n_components=15, min_df=3, seed=42):
    '''文字n-gram TF-IDF + TruncatedSVDでテキスト特徴量を生成（Trainのみでfit）'''
    train_text = train_persona[col].fillna("").astype(str)
    test_text = test_persona[col].fillna("").astype(str)

    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=max_features, min_df=min_df)
    train_tfidf = vectorizer.fit_transform(train_text)
    test_tfidf = vectorizer.transform(test_text)

    n_comp = min(n_components, train_tfidf.shape[1] - 1)
    svd = TruncatedSVD(n_components=n_comp, random_state=seed, algorithm="arpack")
    train_svd = svd.fit_transform(train_tfidf)
    test_svd = svd.transform(test_tfidf)

    col_names = [f"{col}_tfidf_svd_{i}" for i in range(n_comp)]
    train_out = pd.DataFrame(train_svd, columns=col_names)
    train_out[ID_COL] = train_persona[ID_COL].values
    test_out = pd.DataFrame(test_svd, columns=col_names)
    test_out[ID_COL] = test_persona[ID_COL].values
    return train_out, test_out, svd.explained_variance_ratio_.sum()

TEXT_COLS = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック"]

logger.info("テキストTF-IDF+SVD特徴量(A_v1)を生成中...")
tfidf_train_list, tfidf_test_list = [], []
for col in TEXT_COLS:
    tr, te, explained_var = create_tfidf_svd_features(train_persona, test_persona, col, max_features=300, n_components=15, min_df=3, seed=SEED)
    logger.info(f"{col}: SVD累積寄与率={explained_var:.3f}")
    tfidf_train_list.append(tr)
    tfidf_test_list.append(te)

logger.info("テキストTF-IDF+SVD特徴量生成完了")

[2026-08-11 15:40:40] [INFO] テキストTF-IDF+SVD特徴量(A_v1)を生成中...


INFO:39_rejected_blocks_low_noise_recheck:テキストTF-IDF+SVD特徴量(A_v1)を生成中...


[2026-08-11 15:40:41] [INFO] 入社時メモ: SVD累積寄与率=0.760


INFO:39_rejected_blocks_low_noise_recheck:入社時メモ: SVD累積寄与率=0.760


[2026-08-11 15:40:45] [INFO] 上司からのフィードバック: SVD累積寄与率=0.360


INFO:39_rejected_blocks_low_noise_recheck:上司からのフィードバック: SVD累積寄与率=0.360


[2026-08-11 15:40:46] [INFO] 同僚からのフィードバック: SVD累積寄与率=0.421


INFO:39_rejected_blocks_low_noise_recheck:同僚からのフィードバック: SVD累積寄与率=0.421


[2026-08-11 15:40:46] [INFO] テキストTF-IDF+SVD特徴量生成完了


INFO:39_rejected_blocks_low_noise_recheck:テキストTF-IDF+SVD特徴量生成完了


## 3. 四半期/加速度特徴量（D_expanded、`18_`の勝者を継続採用）

`18_`のステップAで、D_expanded（16指標）がD_original（6指標）・Dなしより2 split平均で最良と判明したため、
以降は常にD_expandedを使う（今回はDブロックの再比較は行わない）。

In [11]:
def create_quarterly_features(monthly_df, employee_ids, metrics, suffix=""):
    quarters = {"q1": (0, 5), "q2": (6, 11), "q3": (12, 17), "q4": (18, 23)}
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数")
        features = {"社員ID": employee_id}
        for metric in metrics:
            q_means = {}
            for qname, (lo, hi) in quarters.items():
                vals = emp_data[emp_data["経過月数"].between(lo, hi)][metric]
                q_means[qname] = vals.mean()
                features[f"{metric}_{qname}_mean{suffix}"] = q_means[qname]
            first_half_delta = q_means["q2"] - q_means["q1"] if pd.notna(q_means["q1"]) and pd.notna(q_means["q2"]) else np.nan
            second_half_delta = q_means["q4"] - q_means["q3"] if pd.notna(q_means["q3"]) and pd.notna(q_means["q4"]) else np.nan
            features[f"{metric}_acceleration{suffix}"] = (
                second_half_delta - first_half_delta if pd.notna(first_half_delta) and pd.notna(second_half_delta) else np.nan
            )
        features_list.append(features)
    return pd.DataFrame(features_list)

D_EXPANDED_METRICS = [
    "残業時間", "有給取得日数", "欠勤日数", "研修時間",
    "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
    "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
    "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
    "顧客満足度評価", "担当プロジェクト数", "月例給与_円",
]

logger.info("四半期/加速度特徴量(D_expanded: 16指標)を生成中...")
train_quarterly_exp = create_quarterly_features(train_monthly, train_ids, D_EXPANDED_METRICS, suffix="_exp")
test_quarterly_exp = create_quarterly_features(test_monthly, test_ids, D_EXPANDED_METRICS, suffix="_exp")
logger.info(f"D_expanded: Train {train_quarterly_exp.shape}, Test {test_quarterly_exp.shape}")

[2026-08-11 15:40:47] [INFO] 四半期/加速度特徴量(D_expanded: 16指標)を生成中...


INFO:39_rejected_blocks_low_noise_recheck:四半期/加速度特徴量(D_expanded: 16指標)を生成中...


[2026-08-11 15:42:45] [INFO] D_expanded: Train (2761, 81), Test (2502, 81)


INFO:39_rejected_blocks_low_noise_recheck:D_expanded: Train (2761, 81), Test (2502, 81)


## 4. Persona単位の基本特徴量（split非依存、`18_`と同一）

In [12]:
logger.info("Persona単位の基本特徴量を生成中...")
train_persona["入社日"] = pd.to_datetime(train_persona["入社日"])
test_persona["入社日"] = pd.to_datetime(test_persona["入社日"])

for col in TEXT_COLS:
    train_persona[f"{col}_len"] = train_persona[col].fillna("").astype(str).apply(len)
    test_persona[f"{col}_len"] = test_persona[col].fillna("").astype(str).apply(len)
train_persona["text_total_chars"] = train_persona[TEXT_COLS].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)
test_persona["text_total_chars"] = test_persona[TEXT_COLS].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)

train_persona["入社年"] = train_persona["入社日"].dt.year
train_persona["入社月"] = train_persona["入社日"].dt.month
train_persona["入社四半期"] = train_persona["入社日"].dt.quarter
test_persona["入社年"] = test_persona["入社日"].dt.year
test_persona["入社月"] = test_persona["入社日"].dt.month
test_persona["入社四半期"] = test_persona["入社日"].dt.quarter

train_persona["年齢_x_前職経験"] = train_persona["入社時年齢"] * train_persona["前職経験月数"]
test_persona["年齢_x_前職経験"] = test_persona["入社時年齢"] * test_persona["前職経験月数"]
grade_map = {"G1": 1, "G2": 2, "G3": 3, "G4": 4, "G5": 5}
train_persona["初期等級_num"] = train_persona["初期等級"].map(grade_map)
test_persona["初期等級_num"] = test_persona["初期等級"].map(grade_map)
train_persona["初任給_x_等級"] = train_persona["初任給_円"] * train_persona["初期等級_num"]
test_persona["初任給_x_等級"] = test_persona["初任給_円"] * test_persona["初期等級_num"]

train_persona["is_Q2_新卒"] = ((train_persona["入社四半期"] == 2) & (train_persona["入社区分"] == "新卒")).astype(int)
test_persona["is_Q2_新卒"] = ((test_persona["入社四半期"] == 2) & (test_persona["入社区分"] == "新卒")).astype(int)

logger.info("Persona単位の基本特徴量処理完了")

[2026-08-11 15:42:45] [INFO] Persona単位の基本特徴量を生成中...


INFO:39_rejected_blocks_low_noise_recheck:Persona単位の基本特徴量を生成中...


[2026-08-11 15:42:46] [INFO] Persona単位の基本特徴量処理完了


INFO:39_rejected_blocks_low_noise_recheck:Persona単位の基本特徴量処理完了


## 5. 転居×勤務地マッチの交互作用特徴量（ブロックL、v1=`27_`のPublic確認済み版 / v2=抽出拡張版）

`転居許容`フラグの抽出ロジックは`27_`と同一。`希望勤務地`の抽出のみ2種類を用意する：

- **v1**: `27_`・`25_`・`data_exploration_v3/v4/v5`と同一の正規表現（Public 0.529672で確認済み）
- **v2**: v1に加え、「◯◯を希望。」「◯◯勤務を希望。」「◯◯での勤務を希望。」パターンを追加で
  拾う拡張版。未抽出だった152件（train）を目視確認して発見した言い回し。カバー率が
  88.6%→94.1%（train）/ 95.0%（test）に向上し、ダブル悪条件の該当件数も342→354件に増加、
  効果量はp=1.4×10⁻³⁹→1.7×10⁻⁴³・オッズ比0.189→0.178とむしろ強まった。

In [13]:
def extract_workstyle_section(text):
    if pd.isna(text):
        return None
    m = re.search(r"勤務地・働き方：(.+?)$", text, re.S)
    return m.group(1).strip() if m else None


NEG_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はも]?(許容せず|許容していない|許容しておらず|希望せず|希望しておらず|希望していない)")
POS_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はもを]?(許容し?ており|許容)")


def classify_reloc(s):
    if s is None:
        return None
    if NEG_RELOC.search(s):
        return False
    if POS_RELOC.search(s):
        return True
    return None


def extract_desired_location_v1(s):
    '''27_・25_・EDA v3/v4/v5と同一（Public 0.529672で確認済み、カバー率88.6%/train）'''
    if s is None:
        return None
    m = re.search(r"(?:勤務地は|希望勤務地は)(.+?)(?:を希望|。)", s)
    if m:
        return m.group(1)
    m2 = re.search(r"(.+?)を希望勤務地", s)
    return m2.group(1) if m2 else None


def extract_desired_location_v2(s):
    '''v1に「◯◯(勤務|での勤務)?を希望。」パターンを追加した拡張版（カバー率94.1%/train）'''
    if s is None:
        return None
    loc = extract_desired_location_v1(s)
    if loc is None:
        m3 = re.search(r"^([一-龥ぁ-んァ-ンー]+?)(?:での勤務|勤務)?を希望。", s)
        loc = m3.group(1) if m3 else None
    if loc is not None:
        loc = loc.strip("「」")
    return loc


def create_relocation_mismatch_features(persona_df, extract_fn, state_col, flag_col):
    ws_section = persona_df["入社時メモ"].apply(extract_workstyle_section)
    reloc_ok_raw = ws_section.apply(classify_reloc)
    desired = ws_section.apply(extract_fn)
    actual = persona_df["初期勤務地"]
    match = (desired == actual) & desired.notna()

    # reloc_ok_rawはobject dtype(True/False/None混在)のため、~演算子は使わず
    # 明示的な等価比較でTrue/False/欠損を扱う（欠損に対する~はTypeErrorになる）
    reloc_true = reloc_ok_raw == True
    reloc_false = reloc_ok_raw == False
    valid = desired.notna() & reloc_ok_raw.notna()

    # 4値カテゴリ（決定木が交互作用を直接学習しやすいよう明示的にエンコード）
    state = pd.Series("unknown", index=persona_df.index)
    state[valid & reloc_true & match] = "許容_一致"
    state[valid & reloc_true & ~match] = "許容_不一致"
    state[valid & reloc_false & match] = "非許容_一致"
    state[valid & reloc_false & ~match] = "非許容_不一致"

    # ダブル悪条件フラグ（EDA v5で確認した最も強いシグナル: 転居許容せず AND 勤務地不一致）
    double_bad = (valid & reloc_false & ~match).astype(int)

    return pd.DataFrame({
        "社員ID": persona_df["社員ID"].values,
        state_col: state.values,
        flag_col: double_bad.values,
    })

logger.info("転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...")
train_reloc_v1 = create_relocation_mismatch_features(train_persona, extract_desired_location_v1, "転居x勤務地_状態_v1", "転居x勤務地_ダブル悪条件_v1")
test_reloc_v1 = create_relocation_mismatch_features(test_persona, extract_desired_location_v1, "転居x勤務地_状態_v1", "転居x勤務地_ダブル悪条件_v1")
train_reloc_v2 = create_relocation_mismatch_features(train_persona, extract_desired_location_v2, "転居x勤務地_状態_v2", "転居x勤務地_ダブル悪条件_v2")
test_reloc_v2 = create_relocation_mismatch_features(test_persona, extract_desired_location_v2, "転居x勤務地_状態_v2", "転居x勤務地_ダブル悪条件_v2")

logger.info(f"L_v1: Train {train_reloc_v1.shape}, Test {test_reloc_v1.shape}")
logger.info(f"L_v2: Train {train_reloc_v2.shape}, Test {test_reloc_v2.shape}")
print("L_v1 ダブル悪条件:")
print(train_reloc_v1["転居x勤務地_ダブル悪条件_v1"].value_counts())
print("\nL_v2 ダブル悪条件:")
print(train_reloc_v2["転居x勤務地_ダブル悪条件_v2"].value_counts())

[2026-08-11 15:42:46] [INFO] 転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...


INFO:39_rejected_blocks_low_noise_recheck:転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...


[2026-08-11 15:42:46] [INFO] L_v1: Train (2761, 3), Test (2502, 3)


INFO:39_rejected_blocks_low_noise_recheck:L_v1: Train (2761, 3), Test (2502, 3)


[2026-08-11 15:42:46] [INFO] L_v2: Train (2761, 3), Test (2502, 3)


INFO:39_rejected_blocks_low_noise_recheck:L_v2: Train (2761, 3), Test (2502, 3)


L_v1 ダブル悪条件:
転居x勤務地_ダブル悪条件_v1
0    2419
1     342
Name: count, dtype: int64

L_v2 ダブル悪条件:
転居x勤務地_ダブル悪条件_v2
0    2407
1     354
Name: count, dtype: int64


## 6b. 部署Target Encoding（`15_`〜`37_`と同一）

In [14]:
def create_department_target_encoding(train_persona, test_persona, y_train, fit_ids, seed=42, n_splits=5, smoothing=10):
    '''初期部署IDのKFold + スムージング付きTarget Encoding（15_〜18_の修正版と同一ロジック）'''
    col = "初期部署ID"
    is_fit = train_persona[ID_COL].isin(fit_ids).values
    dept_all = train_persona[col].values
    y_arr = y_train.values
    global_mean = y_arr[is_fit].mean()

    fit_indices = np.where(is_fit)[0]
    dept_fit = dept_all[fit_indices]
    y_fit = y_arr[fit_indices]

    train_te = np.full(len(train_persona), global_mean)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr_idx, val_idx in kf.split(fit_indices):
        df_tr = pd.DataFrame({col: dept_fit[tr_idx], "y": y_fit[tr_idx]})
        stats_tr = df_tr.groupby(col)["y"].agg(["mean", "count"])
        smoothed = (stats_tr["count"] * stats_tr["mean"] + smoothing * global_mean) / (stats_tr["count"] + smoothing)
        mapping = smoothed.to_dict()
        actual_val_idx = fit_indices[val_idx]
        train_te[actual_val_idx] = pd.Series(dept_fit[val_idx]).map(mapping).fillna(global_mean).values

    df_full = pd.DataFrame({col: dept_fit, "y": y_fit})
    stats_full = df_full.groupby(col)["y"].agg(["mean", "count"])
    smoothed_full = (stats_full["count"] * stats_full["mean"] + smoothing * global_mean) / (stats_full["count"] + smoothing)
    mapping_full = smoothed_full.to_dict()
    dept_size_map = stats_full["count"].to_dict()

    not_fit_indices = np.where(~is_fit)[0]
    train_te[not_fit_indices] = pd.Series(dept_all[not_fit_indices]).map(mapping_full).fillna(global_mean).values

    test_te = test_persona[col].map(mapping_full).fillna(global_mean).values

    train_out = pd.DataFrame({
        ID_COL: train_persona[ID_COL].values,
        "dept_target_enc": train_te,
        "dept_size": pd.Series(dept_all).map(dept_size_map).fillna(0).values,
    })
    test_out = pd.DataFrame({
        ID_COL: test_persona[ID_COL].values,
        "dept_target_enc": test_te,
        "dept_size": test_persona[col].map(dept_size_map).fillna(0).values,
    })
    return train_out, test_out

print("✅ 部署Target Encoding関数 定義完了")


✅ 部署Target Encoding関数 定義完了


In [15]:
RESULT_SCHEMA = ["config", "n_features", "val_score", "val_score_all", "val_score_single",
                   "val_single_mean", "val_single_sd", "best_iter", "n_iterations",
                   "params", "submission_path"]

def make_row(**kwargs):
    """全configで同じ列構成のdictを作る。

    28_ は全configが同じキーを持っていたが、37_ は A / BC / D で必要な情報が異なる。
    キー構成がバラバラのままだと、mode="a" でCSVに追記した際に列がずれて壊れるため、
    固定スキーマに揃えてから書き出す。
    """
    unknown = set(kwargs) - set(RESULT_SCHEMA)
    assert not unknown, f"RESULT_SCHEMAに無いキー: {unknown}"
    row = {k: np.nan for k in RESULT_SCHEMA}
    row.update(kwargs)
    return row


def load_checkpoint():
    if CHECKPOINT_PATH.exists():
        return pd.read_csv(CHECKPOINT_PATH)
    return pd.DataFrame(columns=RESULT_SCHEMA)

def save_checkpoint_row(result):
    df = pd.DataFrame([result])[RESULT_SCHEMA]
    write_header = not CHECKPOINT_PATH.exists()
    df.to_csv(CHECKPOINT_PATH, mode="a", header=write_header, index=False)

def run_or_resume(config_label, run_fn):
    checkpoint = load_checkpoint()
    existing = checkpoint[checkpoint["config"] == config_label]
    if len(existing) > 0:
        row = existing.iloc[0].to_dict()
        logger.info(f"[{config_label}] チェックポイントから復元: val_score={row['val_score']}")
        return row
    result = run_fn()
    save_checkpoint_row(result)
    return result

print("✅ チェックポイント関数定義完了（37_版: 固定スキーマで列ずれを防止）")

✅ チェックポイント関数定義完了（37_版: 固定スキーマで列ずれを防止）


In [16]:
SEEDS = [42, 2024, 7, 1234, 99]          # 改善2: シード平均に使う5シード
N_TRIALS = 25                            # 18_〜28_と同一
ITER_SCALE_CANDIDATES = {"x125": 1.25, "x100": 1.00}   # 全件学習時の反復数スケール（2761/2208≒1.25）


def _feature_cols(df):
    return [c for c in df.columns if c not in ["入社日", TARGET_COL]]


def _xy(df, feature_cols):
    return df[feature_cols].fillna(-999), df[TARGET_COL]


def tune_hyperparams(ag_train, ag_val, n_trials=N_TRIALS):
    """Optunaでハイパーパラメータを探索（探索空間は18_〜28_と完全に同一）"""
    feature_cols = _feature_cols(ag_train)
    obj_cols = [c for c in feature_cols if ag_train[c].dtype == "object"]
    X_tr, y_tr = _xy(ag_train, feature_cols)
    X_va, y_va = _xy(ag_val, feature_cols)

    def objective(trial):
        params = {
            "depth": trial.suggest_int("depth", 3, 10),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
            "border_count": trial.suggest_int("border_count", 32, 255),
            "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
            "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
            "iterations": 1000, "random_seed": SEED, "verbose": False,
            "cat_features": obj_cols, "early_stopping_rounds": 50, "task_type": "CPU",
        }
        model = cb.CatBoostClassifier(**params)
        model.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True)
        return log_loss(y_va, model.predict_proba(X_va)[:, 1])

    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=n_trials)
    logger.info(f"  Optuna完了: best_value={study.best_value:.6f}, best_params={study.best_params}")
    return study.best_params


def fit_holdout(ag_train, ag_val, test_features, best_params, seeds):
    """80/20で学習。early stoppingで最良反復数を決め、シードごとの予測を返す"""
    feature_cols = _feature_cols(ag_train)
    obj_cols = [c for c in feature_cols if ag_train[c].dtype == "object"]
    X_tr, y_tr = _xy(ag_train, feature_cols)
    X_va, y_va = _xy(ag_val, feature_cols)
    X_test = test_features[feature_cols].fillna(-999)

    val_preds, test_preds, best_iters = [], [], []
    for seed in seeds:
        model = cb.CatBoostClassifier(
            **best_params, iterations=3000, random_seed=seed, verbose=False,
            cat_features=obj_cols, early_stopping_rounds=100, task_type="CPU",
        )
        model.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True)
        vp = model.predict_proba(X_va)[:, 1]
        val_preds.append(vp)
        test_preds.append(model.predict_proba(X_test)[:, 1])
        best_iters.append(model.get_best_iteration())
        logger.info(f"  seed={seed}: val_logloss={log_loss(y_va, vp):.6f}, best_iteration={best_iters[-1]}")

    return {
        "val_preds": np.array(val_preds), "test_preds": np.array(test_preds),
        "best_iters": best_iters, "y_val": y_va.values, "feature_cols": feature_cols,
    }


def fit_full_train(ag_full, test_features, best_params, n_iterations, seeds):
    """Train全件で学習（改善1）。検証セットが無いので反復数は固定、early stoppingなし"""
    feature_cols = _feature_cols(ag_full)
    obj_cols = [c for c in feature_cols if ag_full[c].dtype == "object"]
    X_tr, y_tr = _xy(ag_full, feature_cols)
    X_test = test_features[feature_cols].fillna(-999)

    test_preds = []
    for seed in seeds:
        model = cb.CatBoostClassifier(
            **best_params, iterations=int(n_iterations), random_seed=seed, verbose=False,
            cat_features=obj_cols, task_type="CPU",
        )
        model.fit(X_tr, y_tr)
        test_preds.append(model.predict_proba(X_test)[:, 1])
        logger.info(f"  seed={seed}: 全件学習完了（iterations={int(n_iterations)}）")
    return np.array(test_preds)


def save_submission(test_index, preds, config_label):
    path = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_{config_label}.csv"
    pd.DataFrame({ID_COL: test_index, TARGET_COL: preds}).to_csv(path, index=False, header=False)
    logger.info(f"  提出ファイル: {path.name}（予測平均={preds.mean():.4f}）")
    return str(path)


print("✅ モデル関数定義完了（tune_hyperparams / fit_holdout / fit_full_train）")

✅ モデル関数定義完了（tune_hyperparams / fit_holdout / fit_full_train）


## 20. 却下済み/未検証ブロックのビルダー関数

**すべて元のノートブックからそのまま移植している**（E/F/G: `20_`、H: `21_`、I: `24_`、J/K: `25_`、
M: `29_`、N/O: `34_`）。今回の目的は「同じ特徴量を、より低ノイズなプロトコルで測り直す」ことなので、
特徴量の中身には一切手を加えない。

`extract_workstyle_section` は `37_` のセル（ブロックL_v2用）で既に定義済みで、`20_`版と
完全に一致することを確認済みのため再定義しない。

In [17]:
# 各ブロックのビルダーは、それぞれを最初に導入したノートブックから**一字一句そのまま**移植している
# （E,F,G: 20_ / H: 21_ / I: 24_ / J,K: 25_ / M: 29_ / N,O: 34_）。
# 再検証の目的は「同じ特徴量を、より低ノイズなプロトコルで測り直す」ことなので、中身は変更しない。

from sklearn.linear_model import LinearRegression

_NEG_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はも]?(許容せず|許容していない|許容しておらず|希望せず|希望しておらず|希望していない)")
_POS_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はもを]?(許容し?ており|許容)")
_NEG_REMOTE = re.compile(r"在宅勤務を(必須条件としていない|希望しない|希望していない|希望せず)")
_POS_REMOTE = re.compile(r"在宅勤務を希望")
ANALYTICAL_MAJOR = {"情報", "理工学"}
ANALYTICAL_JOB = {"IT・エンジニアリング", "データ・商品企画・コンサルティング"}

MOMENTUM_METRICS = [
    "残業時間", "有給取得日数", "欠勤日数", "研修時間",
    "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
    "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
    "360度評価_学習度", "360度評価_共有貢献度",
    "顧客満足度評価", "担当プロジェクト数", "月例給与_円",
]

def extract_memo_section(text, section_name):
    if pd.isna(text):
        return None
    m = re.search(rf"{section_name}：(.+?)(?:\n|$)", text)
    return m.group(1).strip() if m else None


def extract_desired_location(s):
    if s is None:
        return "unknown"
    m = re.search(r"(?:勤務地は|希望勤務地は)(.+?)(?:を希望|。)", s)
    if m:
        return m.group(1)
    m2 = re.search(r"(.+?)を希望勤務地", s)
    return m2.group(1) if m2 else "unknown"


def classify_career_orientation(s):
    if s is None:
        return "unknown"
    if "限定していない" in s or "限定しない" in s:
        return "未定"
    if "管理職" in s:
        return "管理職志向"
    if "専門職" in s:
        return "専門職志向"
    if "安定" in s:
        return "安定志向"
    return "other"


def classify_relocation(s):
    if s is None:
        return "unknown"
    if _NEG_RELOC.search(s):
        return "false"
    if _POS_RELOC.search(s):
        return "true"
    return "unknown"


def classify_remote_pref(s):
    if s is None:
        return "unknown"
    if _POS_REMOTE.search(s):
        return "true"
    if _NEG_REMOTE.search(s):
        return "false"
    return "unknown"


def create_memo_structured_features(persona_df):
    career_section = persona_df["入社時メモ"].apply(lambda t: extract_memo_section(t, "キャリア志向"))
    ws_section = persona_df["入社時メモ"].apply(extract_workstyle_section)
    return pd.DataFrame({
        "社員ID": persona_df["社員ID"].values,
        "memo_career_cat": career_section.apply(classify_career_orientation).values,
        "memo_転居許容": ws_section.apply(classify_relocation).values,
        "memo_在宅希望": ws_section.apply(classify_remote_pref).values,
        "memo_希望勤務地": ws_section.apply(extract_desired_location).values,
    })


def parse_all_study_themes(s):
    if pd.isna(s) or s == "受講なし":
        return [], 0.0
    parts = str(s).split("｜")
    themes, total = [], 0.0
    for part in parts:
        m = re.match(r"(.+?)：([\d.]+)時間", part)
        if m:
            themes.append(m.group(1))
            total += float(m.group(2))
    return themes, total


def create_self_study_features(monthly_df, employee_ids):
    """自己学習実施月数・合計時間・ユニークテーマ数を集計（EDA v3 分析7と同一ロジック）"""
    df = monthly_df[["社員ID", "自己学習（詳細）"]].copy()
    parsed = df["自己学習（詳細）"].apply(parse_all_study_themes)
    df["_themes"] = parsed.apply(lambda x: x[0])
    df["_hours"] = parsed.apply(lambda x: x[1])

    total_hours = df.groupby("社員ID")["_hours"].sum()
    active_months = df[df["_hours"] > 0].groupby("社員ID").size()
    unique_themes = df.groupby("社員ID")["_themes"].apply(lambda s: len(set(t for tl in s for t in tl)))

    out = pd.DataFrame({
        "自己学習合計時間": total_hours,
        "自己学習実施月数": active_months,
        "自己学習ユニークテーマ数": unique_themes,
    })
    out = out.reindex(employee_ids).fillna(0.0).reset_index().rename(columns={"index": "社員ID"})
    return out


def create_engagement_deepdive_features(monthly_df, employee_ids):
    other_eval_cols = ["360度評価_親和度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        info_vals = emp_data["情報共有件数"].values
        is_zero = info_vals == 0
        features["情報共有件数_ゼロ月数"] = int(is_zero.sum())
        max_run = cur_run = 0
        for v in is_zero:
            cur_run = cur_run + 1 if v else 0
            max_run = max(max_run, cur_run)
        features["情報共有件数_最長ゼロ連続月数"] = max_run

        trust_mean = emp_data["360度評価_信頼度"].mean()
        other_mean = emp_data[other_eval_cols].mean(axis=1).mean()
        features["360度評価_信頼度_相対偏差"] = (
            trust_mean - other_mean if pd.notna(trust_mean) and pd.notna(other_mean) else np.nan
        )

        features_list.append(features)
    return pd.DataFrame(features_list)


def create_personal_impression_features(persona_df):
    obs_section = persona_df["入社時メモ"].apply(lambda t: extract_memo_section(t, "人物所見"))
    text = obs_section.fillna("")
    flex = text.apply(lambda t: any(k in t for k in ["柔軟", "切り替え", "適応"]))
    proactive = text.apply(lambda t: any(k in t for k in ["相談", "自ら", "主体的"]))
    plan = text.apply(lambda t: any(k in t for k in ["優先順位", "完了条件", "着実に"]))
    return pd.DataFrame({
        "社員ID": persona_df["社員ID"].values,
        "personal_柔軟性": flex.astype(int).values,
        "personal_主体性相談": proactive.astype(int).values,
        "personal_計画性": plan.astype(int).values,
    })


def create_location_match_features(persona_df):
    ws_section = persona_df["入社時メモ"].apply(extract_workstyle_section)
    desired = ws_section.apply(extract_desired_location)
    actual = persona_df["初期勤務地"]
    match = (desired == actual) & desired.notna()
    # 希望地を抽出できなかった行は「不明」として区別する（一致でも不一致でもない）
    match_cat = pd.Series("unknown", index=persona_df.index)
    match_cat[desired.notna()] = match[desired.notna()].map({True: "match", False: "mismatch"})
    return pd.DataFrame({
        "社員ID": persona_df["社員ID"].values,
        "勤務地希望マッチ": match_cat.values,
    })


def first_raise_month(g):
    g = g.sort_values("経過月数")
    salaries = g["月例給与_円"].values
    months = g["経過月数"].values
    base = salaries[0]
    for i in range(1, len(salaries)):
        if salaries[i] > base:
            return months[i]
    return np.nan


def create_raise_timing_features(monthly_df, employee_ids):
    raise_month = monthly_df.groupby("社員ID", group_keys=False).apply(first_raise_month, include_groups=False)
    raise_month = raise_month.reindex(employee_ids)
    early_flag = (raise_month <= 6).astype(int)
    return pd.DataFrame({
        "社員ID": employee_ids,
        "早期昇給フラグ": early_flag.values,
        "初回昇給月": raise_month.values,
    })


def create_major_job_mismatch_features(persona_df):
    is_analytical_major = persona_df["専攻分野"].isin(ANALYTICAL_MAJOR)
    is_analytical_job = persona_df["初期職種"].isin(ANALYTICAL_JOB)

    # 4値カテゴリ（決定木が交互作用を直接学習しやすいよう明示的にエンコード）
    state = pd.Series("", index=persona_df.index)
    state[is_analytical_major & is_analytical_job] = "分析系専攻_分析系職種"
    state[is_analytical_major & ~is_analytical_job] = "分析系専攻_非分析系職種"
    state[~is_analytical_major & is_analytical_job] = "非分析系専攻_分析系職種"
    state[~is_analytical_major & ~is_analytical_job] = "非分析系専攻_非分析系職種"

    # 片方向ミスマッチフラグ（EDA体系探索で確認した最も強いシグナル:
    # 非分析系専攻の社員が分析系職種に配属された場合。逆方向はほぼ無風）
    mismatch_flag = (~is_analytical_major & is_analytical_job).astype(int)

    return pd.DataFrame({
        "社員ID": persona_df["社員ID"].values,
        "専攻職種_適合状態": state.values,
        "専攻職種_分析ミスマッチ": mismatch_flag.values,
    })


def create_momentum_features(monthly_df, employee_ids, metrics, recent_n=3, prior_n=3):
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数")
        max_month = emp_data["経過月数"].max()
        features = {"社員ID": employee_id}
        for m in metrics:
            recent = emp_data[emp_data["経過月数"] > max_month - recent_n][m].mean()
            prior = emp_data[
                (emp_data["経過月数"] <= max_month - recent_n)
                & (emp_data["経過月数"] > max_month - recent_n - prior_n)
            ][m].mean()
            features[f"{m}_momentum_diff"] = recent - prior if pd.notna(recent) and pd.notna(prior) else np.nan
            features[f"{m}_momentum_ratio"] = (
                recent / prior if pd.notna(recent) and pd.notna(prior) and prior != 0 else np.nan
            )
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_density_features(monthly_df, employee_ids):
    agg = monthly_df.groupby("社員ID").agg(
        残業時間_sum=("残業時間", "sum"),
        研修時間_sum=("研修時間", "sum"),
        情報共有件数_sum=("情報共有件数", "sum"),
        面談回数_sum=("上司との面談実施回数", "sum"),
        在宅勤務日数_sum=("在宅勤務日数", "sum"),
        有給取得日数_sum=("有給取得日数", "sum"),
        欠勤日数_sum=("欠勤日数", "sum"),
        担当プロジェクト数_sum=("担当プロジェクト数", "sum"),
        出勤月数=("経過月数", "count"),
        上司ID_nunique=("上司ID", "nunique"),
    ).reindex(employee_ids)

    out = pd.DataFrame(index=agg.index)
    out["残業時間_per_プロジェクト"] = agg["残業時間_sum"] / agg["担当プロジェクト数_sum"].replace(0, np.nan)
    out["情報共有件数_per_出勤月"] = agg["情報共有件数_sum"] / agg["出勤月数"]
    out["面談回数_per_上司数"] = agg["面談回数_sum"] / agg["上司ID_nunique"].replace(0, np.nan)
    out["研修時間_per_残業時間"] = agg["研修時間_sum"] / agg["残業時間_sum"].replace(0, np.nan)
    out["有給取得_per_欠勤"] = agg["有給取得日数_sum"] / (agg["欠勤日数_sum"] + 0.1)
    out["在宅勤務_per_出勤月"] = agg["在宅勤務日数_sum"] / agg["出勤月数"]
    out["担当プロジェクト_per_出勤月"] = agg["担当プロジェクト数_sum"] / agg["出勤月数"]
    out["情報共有_per_面談"] = agg["情報共有件数_sum"] / (agg["面談回数_sum"] + 0.1)
    out = out.replace([np.inf, -np.inf], np.nan).reset_index().rename(columns={"index": "社員ID"})
    return out


print("✅ 却下済みブロックのビルダー関数 定義完了")


✅ 却下済みブロックのビルダー関数 定義完了


## 21. 各ブロックの特徴量を生成（split非依存）

In [18]:
logger.info("=" * 60); logger.info("却下済み/未検証ブロックの特徴量を生成中...")

train_memo       = create_memo_structured_features(train_persona)      # E
test_memo        = create_memo_structured_features(test_persona)
train_selfstudy  = create_self_study_features(train_monthly, train_ids) # G
test_selfstudy   = create_self_study_features(test_monthly, test_ids)
train_engagement = create_engagement_deepdive_features(train_monthly, train_ids)  # H
test_engagement  = create_engagement_deepdive_features(test_monthly, test_ids)
train_personal   = create_personal_impression_features(train_persona)  # I
test_personal    = create_personal_impression_features(test_persona)
train_locmatch   = create_location_match_features(train_persona)       # J
test_locmatch    = create_location_match_features(test_persona)
train_raise      = create_raise_timing_features(train_monthly, train_ids)  # K
test_raise       = create_raise_timing_features(test_monthly, test_ids)
train_majorjob   = create_major_job_mismatch_features(train_persona)   # M
test_majorjob    = create_major_job_mismatch_features(test_persona)
train_momentum   = create_momentum_features(train_monthly, train_ids, MOMENTUM_METRICS)  # N
test_momentum    = create_momentum_features(test_monthly, test_ids, MOMENTUM_METRICS)
train_density    = create_density_features(train_monthly, train_ids)   # O
test_density     = create_density_features(test_monthly, test_ids)

for nm, df_ in [("E_memo", train_memo), ("G_selfstudy", train_selfstudy), ("H_engagement", train_engagement),
                ("I_impression", train_personal), ("J_locmatch", train_locmatch), ("K_raise", train_raise),
                ("M_majorjob", train_majorjob), ("N_momentum", train_momentum), ("O_density", train_density)]:
    logger.info(f"  {nm:14s}: {df_.shape[1]-1:3d}列, {len(df_)}行")
logger.info("ブロック特徴量の生成完了")

[2026-08-11 15:42:47] [INFO] ============================================================


INFO:39_rejected_blocks_low_noise_recheck:============================================================


[2026-08-11 15:42:47] [INFO] 却下済み/未検証ブロックの特徴量を生成中...


INFO:39_rejected_blocks_low_noise_recheck:却下済み/未検証ブロックの特徴量を生成中...


[2026-08-11 15:44:16] [INFO]   E_memo        :   4列, 2761行


...

INFO:39_rejected_blocks_low_noise_recheck:  N_momentum    :  30列, 2761行


[2026-08-11 15:44:16] [INFO]   O_density     :   8列, 2761行


INFO:39_rejected_blocks_low_noise_recheck:  O_density     :   8列, 2761行


[2026-08-11 15:44:16] [INFO] ブロック特徴量の生成完了


INFO:39_rejected_blocks_low_noise_recheck:ブロック特徴量の生成完了


## 22. `prepare_split`（39_版: 全ブロック対応）

`37_` 版に、却下済みブロック E/F/G/H/I/J/K/M/N/O のマージ処理を足しただけ。
L_v2・全件学習（`split_ratio=1.0`）・検証セットからの早期退職者除外はそのまま引き継ぐ。

Fブロックだけは他と違い、学習期間のIDのみで線形回帰をfitする必要があるため
`prepare_split` の内部に置いている（`20_`と同一のリーク対策）。

In [19]:
def prepare_split(split_ratio, extra_blocks=None, exclude_early_from_val=True):
    '''指定した分割比率で特徴量を組み立てる。

    28_ からの変更点は2つだけ:
      - split_ratio=1.0 を許容（Train全件学習用。ag_tuningは空になる）
      - exclude_early_from_val=True のとき、検証セットから早期退職者を除く（改善3）
    特徴量の作り方そのものは 28_ と完全に同一。
    '''
    extra_blocks = extra_blocks or set()
    sorted_persona = train_persona.sort_values("入社日")
    split_point = int(len(sorted_persona) * split_ratio)
    train_period_ids = set(sorted_persona.iloc[:split_point][ID_COL])

    train_dept_te, test_dept_te = create_department_target_encoding(
        train_persona, test_persona, y_train, fit_ids=train_period_ids, seed=SEED, n_splits=5, smoothing=10
    )

    train_persona_features = train_persona.drop(columns=[TARGET_COL])
    tf = train_persona_features.merge(train_monthly_agg, on=ID_COL, how="left")
    tf = tf.merge(train_cat_change, on=ID_COL, how="left")
    tf = tf.merge(train_missing, on=ID_COL, how="left")
    tf = tf.merge(train_domain, on=ID_COL, how="left")
    tf = tf.merge(train_advanced_stats, on=ID_COL, how="left")
    tf = tf.merge(train_cluster, on=ID_COL, how="left")
    tf = tf.merge(train_dept_te, on=ID_COL, how="left")
    tf = tf.merge(train_eda_feats, on=ID_COL, how="left")
    tf = tf.merge(train_mgr, on=ID_COL, how="left")
    tf = tf.merge(train_quarterly_exp, on=ID_COL, how="left")
    for trdf in tfidf_train_list:
        tf = tf.merge(trdf, on=ID_COL, how="left")

    ttf = test_persona.merge(test_monthly_agg, on=ID_COL, how="left")
    ttf = ttf.merge(test_cat_change, on=ID_COL, how="left")
    ttf = ttf.merge(test_missing, on=ID_COL, how="left")
    ttf = ttf.merge(test_domain, on=ID_COL, how="left")
    ttf = ttf.merge(test_advanced_stats, on=ID_COL, how="left")
    ttf = ttf.merge(test_cluster, on=ID_COL, how="left")
    ttf = ttf.merge(test_dept_te, on=ID_COL, how="left")
    ttf = ttf.merge(test_eda_feats, on=ID_COL, how="left")
    ttf = ttf.merge(test_mgr, on=ID_COL, how="left")
    ttf = ttf.merge(test_quarterly_exp, on=ID_COL, how="left")
    for tedf in tfidf_test_list:
        ttf = ttf.merge(tedf, on=ID_COL, how="left")

    if "L1" in extra_blocks:
        tf = tf.merge(train_reloc_v1, on=ID_COL, how="left")
        ttf = ttf.merge(test_reloc_v1, on=ID_COL, how="left")

    if "L2" in extra_blocks:
        tf = tf.merge(train_reloc_v2, on=ID_COL, how="left")
        ttf = ttf.merge(test_reloc_v2, on=ID_COL, how="left")

    if "E" in extra_blocks:
        tf = tf.merge(train_memo, on=ID_COL, how="left")
        ttf = ttf.merge(test_memo, on=ID_COL, how="left")

    if "G" in extra_blocks:
        tf = tf.merge(train_selfstudy, on=ID_COL, how="left")
        ttf = ttf.merge(test_selfstudy, on=ID_COL, how="left")

    if "H" in extra_blocks:
        tf = tf.merge(train_engagement, on=ID_COL, how="left")
        ttf = ttf.merge(test_engagement, on=ID_COL, how="left")

    if "I" in extra_blocks:
        tf = tf.merge(train_personal, on=ID_COL, how="left")
        ttf = ttf.merge(test_personal, on=ID_COL, how="left")

    if "J" in extra_blocks:
        tf = tf.merge(train_locmatch, on=ID_COL, how="left")
        ttf = ttf.merge(test_locmatch, on=ID_COL, how="left")

    if "K" in extra_blocks:
        tf = tf.merge(train_raise, on=ID_COL, how="left")
        ttf = ttf.merge(test_raise, on=ID_COL, how="left")

    if "M" in extra_blocks:
        tf = tf.merge(train_majorjob, on=ID_COL, how="left")
        ttf = ttf.merge(test_majorjob, on=ID_COL, how="left")

    if "N" in extra_blocks:
        tf = tf.merge(train_momentum, on=ID_COL, how="left")
        ttf = ttf.merge(test_momentum, on=ID_COL, how="left")

    if "O" in extra_blocks:
        tf = tf.merge(train_density, on=ID_COL, how="left")
        ttf = ttf.merge(test_density, on=ID_COL, how="left")

    # F: 中途入社者の「前職経験月数から予測される等級」からの残差（20_と同一。学習期間のIDのみでfit）
    if "F" in extra_blocks:
        mid_fit = tf[(tf[ID_COL].isin(train_period_ids)) & (tf["入社区分"] == "中途")]
        reg = LinearRegression().fit(mid_fit[["前職経験月数"]].values, mid_fit["初期等級_num"].values)
        for df_ in [tf, ttf]:
            is_mid = (df_["入社区分"] == "中途")
            df_["経験等級_残差"] = np.nan
            if is_mid.sum() > 0:
                df_.loc[is_mid, "経験等級_残差"] = (
                    df_.loc[is_mid, "初期等級_num"].values - reg.predict(df_.loc[is_mid, ["前職経験月数"]].values)
                )
            df_["is_中途"] = is_mid.astype(int)

    _train_period_features = tf[tf[ID_COL].isin(train_period_ids)]
    job_dev_metrics = ["残業時間_mean", "研修時間_mean", "360度評価_親和度_mean"]
    job_means = {m: _train_period_features.groupby("初期職種")[m].mean().to_dict() for m in job_dev_metrics}
    category_means_train = {m: _train_period_features.groupby("入社区分")[m].mean().to_dict() for m in job_dev_metrics}
    grade_salary_mean = _train_period_features.groupby("初期等級")["初任給_円"].mean().to_dict()
    category_salary_mean = _train_period_features.groupby("入社区分")["初任給_円"].mean().to_dict()
    grade_monthly_salary_mean = _train_period_features.groupby("初期等級")["月例給与_円_mean"].mean().to_dict()

    for df_ in [tf, ttf]:
        for m in job_dev_metrics:
            df_[f"{m}_job_deviation"] = df_[m] - df_["初期職種"].map(job_means[m])
        df_["研修時間_職種比"] = df_["研修時間_mean"] / df_["初期職種"].map(job_means["研修時間_mean"]).replace(0, np.nan)
        df_["研修時間_区分比"] = df_["研修時間_mean"] / df_["入社区分"].map(category_means_train["研修時間_mean"]).replace(0, np.nan)
        df_["初任給_等級内偏差"] = df_["初任給_円"] - df_["初期等級"].map(grade_salary_mean)
        df_["初任給_区分内偏差"] = df_["初任給_円"] - df_["入社区分"].map(category_salary_mean)
        df_["月例給与_等級内偏差"] = df_["月例給与_円_mean"] - df_["初期等級"].map(grade_monthly_salary_mean)

    drop_cols = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック",
                 "初期部署ID", "初期等級", "最終学歴", "前職職種"]
    tf = tf.drop(columns=[c for c in drop_cols if c in tf.columns]).set_index(ID_COL)
    ttf = ttf.drop(columns=[c for c in drop_cols if c in ttf.columns]).set_index(ID_COL)

    target_series = train_persona.set_index(ID_COL)[TARGET_COL]
    tf_sorted = tf.sort_values("入社日")
    y_sorted = target_series.loc[tf_sorted.index]

    ag_train = tf_sorted.iloc[:split_point].copy()
    ag_tuning = tf_sorted.iloc[split_point:].copy()
    ag_train[TARGET_COL] = y_sorted.iloc[:split_point].values
    ag_tuning[TARGET_COL] = y_sorted.iloc[split_point:].values

    # --- 改善3: 検証セットから早期退職者を除く（学習側からは除かない） ---
    if exclude_early_from_val and len(ag_tuning) > 0:
        n_before = len(ag_tuning)
        ag_tuning = ag_tuning[~ag_tuning.index.isin(EARLY_LEAVER_IDS)]
        logger.info(f"  検証セット: {n_before} → {len(ag_tuning)}件（早期退職者{n_before - len(ag_tuning)}名を除外）")

    return ag_train, ag_tuning, ttf

print("✅ prepare_split定義完了（39_版: 却下済みブロックE/F/G/H/I/J/K/M/N/Oに対応）")

✅ prepare_split定義完了（39_版: 却下済みブロックE/F/G/H/I/J/K/M/N/Oに対応）


## 23. アブレーション（ハイパーパラメータ固定・5シード・2 split）

`A_PARAMS` を全構成で固定するのが旧プロトコルとの最大の違い。これにより
「ブロックの効果」と「Optunaの着地点のブレ」の交絡が消える。

## 22b. この検証系の「分解能」を先に把握しておく

判定を始める前に、**そもそもこの検証セットでどの程度の効果量まで見分けられるのか**を、
既存の保存済み検証予測（`data/output/20260811/*_valpreds.npy`）で較正した。
ペアードブートストラップ（n=553、2000回）の結果:

| 比較 | 実測diff | 95%信頼区間 | 判定 |
|---|---|---|---|
| **ベースライン vs L_v2**（Publicで確認済みの本物の改善） | **-0.0512** | **[-0.0723, -0.0325]** | **有意に改善** |
| ベースライン vs N_momentum（`34_`で不採用） | -0.0006 | [-0.0128, +0.0109] | 0を跨ぐ |
| ベースライン vs O_density（`34_`で不採用） | -0.0014 | [-0.0119, +0.0093] | 0を跨ぐ |

**信頼区間の半幅は約 ±0.011**。つまりこの検証系は:

- **L級の効果（val で -0.05）は確実に検出できる**
- **0.005前後の効果は原理的に検出できない** ——シードを何個平均しても変わらない。
  シード平均が減らせるのは*シードのばらつき*だけで、**検証標本そのもののゆらぎ**（553名しかいない）は減らない

### この事実が意味すること

`20_`〜`35_` が追いかけていた効果量（L_v1→L_v2 の Public差 0.0002、N・O の ±0.001）は、
**どんなプロトコルを組んでも Train から作った検証セットでは測れない領域にある**。
`39_` で「CIが0を跨ぐ」という結果が出た場合、それは「効果がない」ではなく
**「この方法では測定できない」**と読むべきである。

したがって本ノートブックの現実的な達成目標は次の2つになる。

1. **A群（却下済みブロック）が L 級の効果を持っていないことを確定させる** → 持っていなければ、
   ablation ベースの特徴量探索はこれ以上やっても情報が得られないと結論できる
2. **B群（E・J との組み合わせ）に L 級の効果があるかを見る** → E単体・J単体は 18_ベースライン比で
   Public 0.0155 / 0.0097 の改善実績があり、L級に近い。検出できる可能性がある

`control_noL`（Lブロックを外した構成）を**陽性対照**として入れてあるのはこのため。
これが「有意に悪化」と判定されなければ測定系自体が壊れているので、他の結果も信用してはいけない。

In [20]:
# 37_ config A で選ばれた設定（= 28_ と同一。Public 0.529454 / 全件学習版D3で 0.522659 の実績）
A_PARAMS = {
    "depth": 4, "learning_rate": 0.03518359458951149, "l2_leaf_reg": 2.217690447016724,
    "border_count": 218, "bagging_temperature": 0.6787467566574921, "random_strength": 1.438494697238285,
}

# 4つの時系列split。検証は常に「直近側」に置き、切り出し位置だけを変える
# （Trainの終盤ほどTest期間に近いため、Test直前の期間を検証に使う構造は維持する）
SPLIT_RATIOS = {"split_85_15": 0.85, "split_80_20": 0.80, "split_75_25": 0.75, "split_70_30": 0.70}
SEEDS_ABL = [42, 2024, 7, 1234, 99, 314, 2718, 577]     # 8シード

# --- 38_ の知見: early stopping をやめて反復数を固定する ---
# 38_ の感度曲線（80/20, n_train=2208）で、固定反復数の最適点は 560 だった。
# 一方 early stopping の best_iteration は 448.6 で、真の最適点を系統的に下回っていた
# （ノイズのある曲線の最初の谷で止まるため。80%学習で 0.518820 → 0.517498 の差）。
#
# アブレーションで early stopping を使うと、構成ごとに停止位置がバラつき、
# 「ブロックの効果」と「反復数の選ばれ方」が交絡する。反復数を固定すれば、
# 構成間で変わるのは特徴量ブロックだけになる。
#
# 曲線は 350〜900 で振れ幅 0.0035（シードsd 0.0044 以下）と平坦なので、
# 学習件数に比例させたスケーリングで十分安全。
ITER_AT_2208 = 560          # 38_ の感度曲線の最適点（n_train=2208 のとき）

def iters_for(n_train):
    """学習件数に比例させて反復数を決める（38_の最適点560 @ 2208 を基準にスケール）"""
    return max(int(round(ITER_AT_2208 * n_train / 2208)), 50)


def fit_holdout_fixed(ag_train, ag_val, params, n_iter, seeds):
    """反復数を固定して学習（early stoppingなし）。構成間で停止位置がブレないようにする"""
    feature_cols = _feature_cols(ag_train)
    obj_cols = [c for c in feature_cols if ag_train[c].dtype == "object"]
    X_tr, y_tr = _xy(ag_train, feature_cols)
    X_va, y_va = _xy(ag_val, feature_cols)
    val_preds = []
    for seed in seeds:
        m = cb.CatBoostClassifier(**params, iterations=int(n_iter), random_seed=seed,
                                  verbose=False, cat_features=obj_cols, task_type="CPU")
        m.fit(X_tr, y_tr)
        val_preds.append(m.predict_proba(X_va)[:, 1])
    return {"val_preds": np.array(val_preds), "y_val": y_va.values, "feature_cols": feature_cols}

BLOCK_CONFIGS = {
    "baseline_Lv2":  {"L2"},                    # 現在の最良の特徴量セット（比較の基準）
    # --- 陽性対照: Lブロックを外した構成。Lは Public 0.550352→0.529454 の実績があるので、
    #     この構成は「有意に悪化」と判定されなければならない。されなければ測定系が壊れている ---
    "control_noL":   set(),
    # --- A群: 却下済みブロックの再検証 ---
    "Lv2_plus_F":    {"L2", "F"},
    "Lv2_plus_G":    {"L2", "G"},
    "Lv2_plus_H":    {"L2", "H"},
    "Lv2_plus_I":    {"L2", "I"},
    "Lv2_plus_K":    {"L2", "K"},
    "Lv2_plus_M":    {"L2", "M"},
    "Lv2_plus_N":    {"L2", "N"},
    "Lv2_plus_O":    {"L2", "O"},
    # --- B群: 確認済みブロックの組み合わせ（未検証） ---
    "Lv2_plus_E":    {"L2", "E"},
    "Lv2_plus_J":    {"L2", "J"},
    "Lv2_plus_EJ":   {"L2", "E", "J"},
}

# 検証予測を日付非依存パスに保存する（ブートストラップ用。チェックポイントから再開しても読める）
VALPRED_DIR = CHECKPOINT_DIR / f"{SCRIPT_NAME}_valpreds"
VALPRED_DIR.mkdir(parents=True, exist_ok=True)


def make_ablation_run(config_label, ratio, blocks):
    def _run():
        logger.info("=" * 60); logger.info(f"[{config_label}] blocks={sorted(blocks)}")
        ag_tr, ag_va, test_feat = prepare_split(ratio, extra_blocks=blocks, exclude_early_from_val=True)
        n_iter = iters_for(len(ag_tr))
        out = fit_holdout_fixed(ag_tr, ag_va, A_PARAMS, n_iter, seeds=SEEDS_ABL)
        single = [log_loss(out["y_val"], vp) for vp in out["val_preds"]]
        avg_pred = out["val_preds"].mean(axis=0)
        avg = log_loss(out["y_val"], avg_pred)
        # ブートストラップ用に「シード平均した検証予測」と正解ラベルを保存
        np.save(VALPRED_DIR / f"{config_label}_pred.npy", avg_pred)
        np.save(VALPRED_DIR / f"{config_label}_y.npy", out["y_val"])
        logger.info(f"  n_train={len(ag_tr)}, n_val={len(ag_va)}, iterations={n_iter}")
        logger.info(f"  単一シード平均={np.mean(single):.6f} (sd={np.std(single):.6f}), "
                    f"シード平均={avg:.6f}, n_features={len(out['feature_cols'])}")
        return make_row(config=config_label, n_features=len(out["feature_cols"]),
                        val_score=avg, val_score_single=single[0],
                        val_single_mean=float(np.mean(single)), val_single_sd=float(np.std(single)),
                        best_iter=float(n_iter), params=json.dumps(sorted(blocks)),
                        submission_path="(アブレーションのみ)")
    return _run

ablation = []
for split_name, ratio in SPLIT_RATIOS.items():
    for block_name, blocks in BLOCK_CONFIGS.items():
        label = f"{split_name}_{block_name}"
        r = run_or_resume(label, make_ablation_run(label, ratio, blocks))
        ablation.append({"split": split_name, "block_config": block_name,
                         "n_features": int(r["n_features"]),
                         "val_単一平均": float(r["val_single_mean"]), "val_sd": float(r["val_single_sd"]),
                         "val_シード平均": float(r["val_score"]), "best_iter": float(r["best_iter"])})

ablation_df = pd.DataFrame(ablation)
ablation_df.to_csv(CHECKPOINT_DIR / f"{SCRIPT_NAME}_ablation.csv", index=False)
display(ablation_df.round(6))
print(f"\n典型的なシードsd: {ablation_df['val_sd'].mean():.6f}")

# 各splitの検証規模はログに記録済み（n_val）。実際の信頼区間は後段のブートストラップで求める。

[2026-08-11 15:44:17] [INFO] ============================================================


INFO:39_rejected_blocks_low_noise_recheck:============================================================


[2026-08-11 15:44:17] [INFO] [split_85_15_baseline_Lv2] blocks=['L2']


INFO:39_rejected_blocks_low_noise_recheck:[split_85_15_baseline_Lv2] blocks=['L2']


[2026-08-11 15:44:17] [INFO]   検証セット: 415 → 401件（早期退職者14名を除外）


...

INFO:39_rejected_blocks_low_noise_recheck:  n_train=1932, n_val=798, iterations=490


[2026-08-11 16:15:28] [INFO]   単一シード平均=0.533856 (sd=0.006786), シード平均=0.530431, n_features=446


INFO:39_rejected_blocks_low_noise_recheck:  単一シード平均=0.533856 (sd=0.006786), シード平均=0.530431, n_features=446


,split,block_config,n_features,val_単一平均,val_sd,val_シード平均,best_iter
0,split_85_15,baseline_Lv2,441,0.518828,0.006033,0.515024,595.0
1,split_85_15,control_noL,439,0.547663,0.006253,0.543609,595.0
2,split_85_15,Lv2_plus_F,443,0.514546,0.004865,0.511061,595.0
3,split_85_15,Lv2_plus_G,444,0.506122,0.006248,0.502623,595.0
4,split_85_15,Lv2_plus_H,444,0.511571,0.006036,0.508191,595.0
5,split_85_15,Lv2_plus_I,444,0.517108,0.004139,0.513575,595.0
6,split_85_15,Lv2_plus_K,443,0.513650,0.002707,0.510090,595.0
7,split_85_15,Lv2_plus_M,443,0.510106,0.006946,0.506570,595.0
8,split_85_15,Lv2_plus_N,471,0.517179,0.004945,0.513820,595.0
9,split_85_15,Lv2_plus_O,449,0.516453,0.005537,0.512881,595.0



典型的なシードsd: 0.004887


## 24. 結果の集計と判定

In [21]:
piv = ablation_df.pivot(index="block_config", columns="split", values="val_シード平均")
piv = piv[list(SPLIT_RATIOS)]
piv["mean"] = piv.mean(axis=1)
base_row = piv.loc["baseline_Lv2"]
piv["mean_diff"] = piv["mean"] - base_row["mean"]
piv["改善split数"] = [(piv.loc[i, list(SPLIT_RATIOS)] < base_row[list(SPLIT_RATIOS)]).sum() for i in piv.index]
piv = piv.sort_values("mean_diff")
display(piv.round(6))


def paired_bootstrap(config_label, split_name, n_boot=2000, seed=0):
    """同一の検証社員上で「構成 − ベースライン」のlogloss差をブートストラップする。

    構成とベースラインは同じ検証社員で評価されているので、社員をリサンプリングすれば
    差の分布が直接得られる（ペアード＝対応のあるブートストラップ）。
    """
    d = VALPRED_DIR
    y = np.load(d / f"{split_name}_{config_label}_y.npy")
    p_cfg = np.load(d / f"{split_name}_{config_label}_pred.npy")
    p_base = np.load(d / f"{split_name}_baseline_Lv2_pred.npy")
    rng = np.random.default_rng(seed)
    n = len(y); diffs = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, n)
        yy = y[idx]
        if yy.min() == yy.max():       # 片方のクラスしか出なかった標本は捨てる
            diffs[b] = np.nan; continue
        diffs[b] = log_loss(yy, p_cfg[idx], labels=[0, 1]) - log_loss(yy, p_base[idx], labels=[0, 1])
    diffs = diffs[~np.isnan(diffs)]
    return float(np.mean(diffs)), float(np.percentile(diffs, 2.5)), float(np.percentile(diffs, 97.5))


boot_rows = []
for block_name in BLOCK_CONFIGS:
    if block_name == "baseline_Lv2":
        continue
    per_split = {}
    for split_name in SPLIT_RATIOS:
        m, lo, hi = paired_bootstrap(block_name, split_name)
        per_split[split_name] = (m, lo, hi)
    n_ci_neg = sum(1 for (m, lo, hi) in per_split.values() if hi < 0)   # 95%CIが完全に0未満
    n_ci_pos = sum(1 for (m, lo, hi) in per_split.values() if lo > 0)   # 95%CIが完全に0超（悪化）
    boot_rows.append({
        "block_config": block_name,
        "平均diff": float(np.mean([v[0] for v in per_split.values()])),
        "CI下限の最大": float(max(v[1] for v in per_split.values())),
        "CI上限の最小": float(min(v[2] for v in per_split.values())),
        "改善が有意なsplit数": n_ci_neg,
        "悪化が有意なsplit数": n_ci_pos,
    })

boot_df = pd.DataFrame(boot_rows).sort_values("平均diff")
display(boot_df.round(6))
boot_df.to_csv(CHECKPOINT_DIR / f"{SCRIPT_NAME}_bootstrap.csv", index=False)

# --- 採否の判定 ---
# 条件: 4 splitすべてで平均diffが負、かつ 95%CIが完全に0未満のsplitが2つ以上
winners_idx = []
for r in boot_rows:
    bn = r["block_config"]
    all_neg = all(piv.loc[bn, s] < base_row[s] for s in SPLIT_RATIOS)
    if all_neg and r["改善が有意なsplit数"] >= 2:
        winners_idx.append(bn)
winners = piv.loc[winners_idx] if winners_idx else piv.iloc[0:0]

# --- 陽性対照のチェック（これが通らなければ他の結果も信用しない） ---
ctrl = boot_df[boot_df.block_config == "control_noL"].iloc[0]
ctrl_ok = ctrl["悪化が有意なsplit数"] >= 2
print("\n【陽性対照】control_noL（Lブロックを外した構成）")
print(f"  平均diff={ctrl['平均diff']:+.5f}, 悪化が有意なsplit数={int(ctrl['悪化が有意なsplit数'])}/{len(SPLIT_RATIOS)}")
print(f"  → {'OK: 測定系はL級の効果を検出できている' if ctrl_ok else '⚠ NG: 測定系が期待通り動いていない。以降の結果は信用しないこと'}")

print("\n【判定条件】4 splitすべてで改善 かつ 95%CIが完全に0未満のsplitが2つ以上")
if len(winners) == 0:
    print("  → 該当なし。")
    print("     これは『効果がない』ではなく『この検証系（CI半幅±0.011）では測定できない』が正しい読み方。")
    print("     A群がL級の効果を持たないことが確定したので、ablationベースの特徴量探索は打ち切ってよい。")
else:
    print(f"  → 提出候補: {list(winners.index)}")

split,split_85_15,split_80_20,split_75_25,split_70_30,mean,mean_diff,改善split数
block_config,,,,,,,
Lv2_plus_G,0.502623,0.504922,0.524917,0.518889,0.512838,-0.010950,4
Lv2_plus_M,0.506570,0.513593,0.532520,0.526164,0.519712,-0.004076,4
Lv2_plus_K,0.510090,0.513607,0.531240,0.527311,0.520562,-0.003226,3
Lv2_plus_H,0.508191,0.516103,0.533858,0.527372,0.521381,-0.002407,3
Lv2_plus_F,0.511061,0.514921,0.534895,0.525896,0.521693,-0.002095,3
Lv2_plus_O,0.512881,0.514199,0.534916,0.525461,0.521864,-0.001924,3
Lv2_plus_I,0.513575,0.514590,0.533179,0.527211,0.522139,-0.001649,3
Lv2_plus_J,0.510677,0.517781,0.532821,0.527685,0.522241,-0.001547,3
Lv2_plus_E,0.511065,0.517026,0.533877,0.527236,0.522301,-0.001487,3


,block_config,平均diff,CI下限の最大,CI上限の最小,改善が有意なsplit数,悪化が有意なsplit数
2,Lv2_plus_G,-0.010930,-0.012310,-0.006856,4,0
6,Lv2_plus_M,-0.004056,-0.006356,-0.000684,1,0
5,Lv2_plus_K,-0.003246,-0.001875,-0.001798,3,0
3,Lv2_plus_H,-0.002395,-0.001931,-0.002194,1,0
1,Lv2_plus_F,-0.002066,-0.003358,-0.000260,1,0
8,Lv2_plus_O,-0.001896,-0.003478,-0.000350,1,0
4,Lv2_plus_I,-0.001619,-0.002078,-0.000563,1,0
10,Lv2_plus_J,-0.001564,-0.001649,0.000078,0,0
9,Lv2_plus_E,-0.001486,-0.002072,0.000019,0,0
11,Lv2_plus_EJ,-0.000185,0.001009,0.000810,0,1



【陽性対照】control_noL（Lブロックを外した構成）
  平均diff=+0.03834, 悪化が有意なsplit数=4/4
  → OK: 測定系はL級の効果を検出できている

【判定条件】4 splitすべてで改善 かつ 95%CIが完全に0未満のsplitが2つ以上
  → 提出候補: ['Lv2_plus_G']


## 25. 提出候補の全件学習（該当があった場合のみ）

`37_` で確定した通り、提出は**全件学習＋シード平均**で行う。反復数は80/20の `best_iteration` 平均を
件数比1.25倍したもの。

In [22]:
# 38_ の感度曲線（最適点560 @ n_train=2208）を全件2761へスケール。
# 38_ H1 が採用したのと同じ 700。37_ D3 の 560 は early stopping 由来でやや学習不足だった。
ITER_FULL = iters_for(2761)   # = 700

def make_full_sub(block_name, blocks, n_iter):
    def _run():
        logger.info("=" * 60); logger.info(f"[full_{block_name}] 全件学習 iterations={n_iter}, blocks={sorted(blocks)}")
        ag_full_b, _, test_feat_full = prepare_split(1.0, extra_blocks=blocks, exclude_early_from_val=True)
        preds = fit_full_train(ag_full_b, test_feat_full, A_PARAMS, n_iter, seeds=SEEDS)
        path = save_submission(test_feat_full.index, preds.mean(axis=0), f"full_{block_name}")
        return make_row(config=f"full_{block_name}", n_features=len(_feature_cols(ag_full_b)),
                        n_iterations=n_iter, params=json.dumps(sorted(blocks)), submission_path=path)
    return _run

submitted = []
for block_name in winners.index:
    r = run_or_resume(f"full_{block_name}",
                      make_full_sub(block_name, BLOCK_CONFIGS[block_name], ITER_FULL))
    submitted.append({"block_config": block_name, "iterations": ITER_FULL,
                      "submission": Path(r["submission_path"]).name})

if submitted:
    display(pd.DataFrame(submitted))
else:
    print("提出候補なし。全件学習はスキップした。")
    print("現時点の最良は引き続き 37_ D3_Aparams_full_x125（Public 0.522659）。")

[2026-08-11 16:17:54] [INFO] ============================================================


INFO:39_rejected_blocks_low_noise_recheck:============================================================


[2026-08-11 16:17:54] [INFO] [full_Lv2_plus_G] 全件学習 iterations=700, blocks=['G', 'L2']


INFO:39_rejected_blocks_low_noise_recheck:[full_Lv2_plus_G] 全件学習 iterations=700, blocks=['G', 'L2']


[2026-08-11 16:18:00] [INFO]   seed=42: 全件学習完了（iterations=700）


...

[2026-08-11 16:18:24] [INFO]   seed=99: 全件学習完了（iterations=700）


INFO:39_rejected_blocks_low_noise_recheck:  seed=99: 全件学習完了（iterations=700）


[2026-08-11 16:18:24] [INFO]   提出ファイル: 20260811_39_rejected_blocks_low_noise_recheck_full_Lv2_plus_G.csv（予測平均=0.5864）


INFO:39_rejected_blocks_low_noise_recheck:  提出ファイル: 20260811_39_rejected_blocks_low_noise_recheck_full_Lv2_plus_G.csv（予測平均=0.5864）


,block_config,iterations,submission
0,Lv2_plus_G,700,20260811_39_rejected_blocks_low_noise_recheck_...


## 26. 解釈の指針

### なぜブートストラップで判定するのか

`26_` のブロックKは「両split一貫して改善」という基準を満たしながら Public で3.5%悪化した。
一貫性は「同じ方向にズレたか」しか見ておらず、**ズレの大きさがノイズを超えているか**を問わない。
ペアードブートストラップの信頼区間はそこを直接測る。

判定条件を「4 splitすべてで改善」かつ「95%CIが完全に0未満のsplitが2つ以上」と置いたのは、
片方だけでは弱いため。前者は方向の一貫性、後者は効果量の大きさを見ている。

### 「該当なし」でも収穫である

EDA v6 第4節から、A群の行動系（G・H・K・N・O）は**理論的に不利**だと分かっている
——行動系が予測しているのは短期離職で、Test には短期離職者が1人もいない。
この分解能でも改善が出ないなら、「旧プロトコルがノイズで誤判定していた」のではなく
**本当に効かない**ことが確定し、この方向の探索を安心して打ち切れる。

### B群が改善した場合の注意

E・J・L は3つとも単体で Public 改善が確認済みだが、`combo_EFG`・`combo_EG`・`combo_EI` は
いずれも検証で改善して見えて Public で悪化した。**検証で勝っただけでは採用しない**。
提出して Public で確認するまでは、`37_` D3（Public 0.522659）が最良のままとする。

ただし今回は分解能が上がっているぶん、過去の combo 失敗よりは信用してよい。
特に `27_` の combo_JL は「Lとほぼ同水準（差0.0003）」で見送られたが、
その差は当時のシードsd（0.005前後）に完全に埋もれていた。**測れていなかっただけ**の可能性がある。

### 次のアクション

- 提出候補が出た場合は Public で確認し、`data/output/submit_result_report.md` に追記する。
- 該当なしだった場合、**特徴量追加の方向は打ち切り**。EDA v6 で
  テキスト（第7節）・交絡候補（第8節）・行動系（第4節）・生存時間の活用（優先度4）は
  すべて枯渇を確認済みなので、残るのは `38_` のハイパーパラメータ最適化のみになる。